# E. coli reduced-pathway batch: editable native PharmaPy workflow

This notebook contains the tasks performed by the example scripts directly: configuration, native construction, forward simulation, reporting, plotting, inference, and regression checks. It does **not** import or execute `run_*.py` helpers. Reusable modeling, estimation, and experiment-design algorithms remain in `PharmaPy`.

Use the configured **Python 3.11** environment described in the parent README as the notebook kernel. The complete design study performs repeated fits and refinement checks and can take **tens of minutes**. Outputs are held in memory by default; the last step can export to a separate destination.

For a new example, copy this notebook and compatible `inputs/` files, change the paths and configuration steps, and provide scientifically justified model/property definitions. A new organism is not inferred from its name. These templates assume a declared **constant observation density**. Synthetic observations demonstrate methodology, not independent experimental validation.

The standalone scripts remain available for command-line reproduction. Their original thresholds, seeds, and calculations are retained here; comparisons against the distributed outputs are automatically skipped for changed configurations.

**Forward simulation only:** execute Steps 1–6 and stop. Continue to inference only after reviewing the separate estimation/design settings; those studies have their own operating overrides and sampling times.


## Step 1 — Imports and input locations

Set example_dir to your copied example folder when adapting this notebook. The repository must be importable; the automatic search works when opened inside this checkout. No example files are changed.

In [2]:
# STEP 1: Imports and input locations
# Set example_dir to your copied example folder when adapting this notebook.
# The repository must be importable; the automatic search works when opened inside this
# checkout. No example files are changed.

from pathlib import Path
from copy import deepcopy
import csv
import hashlib
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

# If the notebook is outside this checkout, set repo_root explicitly instead.
repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / "PharmaPy").is_dir()), None)
if repo_root is None:
    raise ValueError("Set repo_root to your PharmaPy checkout before running this cell")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from PharmaPy.Bioreactors import (
    build_bioreactor, BioreactorEstimationProblem, ObservationSeries,
    estimate_bioreactor_parameters,
)
from PharmaPy.Bioreactors.experiments import native_experiment_model, run_design_study

example_dir = repo_root / "examples/bioreactors/batch_ecoli_dfba"
# For a copied example, replace the line above with Path("/absolute/path/to/my_case").
reference_dir = repo_root / "examples/bioreactors/batch_ecoli_dfba"
input_dir = example_dir / "inputs"
thermo_path = input_dir / "thermo.json"
WRITE_ARTIFACTS = False
EXPORT_DIR = example_dir / "custom_results"
COMPARE_CANONICAL = True


## Step 2 — Load and edit model and study configuration

Edit the dictionaries here or the JSON files before running. Keep mechanism identifiers, species/property mappings, recipe names, and duplicated initial volumes consistent. Initial pathway amounts are mol and biomass kgDW; culture amounts are mmol, cells million cells, product g. Update study times, parameter paths, noise, and candidate settings when changing biology. Example overrides below are commented out.

In [4]:
# STEP 2: Load and edit model and study configuration
# Edit the dictionaries here or the JSON files before running. Keep
# mechanism identifiers, species/property mappings, recipe names, and duplicated initial
# volumes consistent. Initial pathway amounts are mol and biomass kgDW; culture amounts
# are mmol, cells million cells, product g. Update study times, parameter paths, noise,
# and candidate settings when changing biology. Example overrides below are commented
# out.

case = json.loads((input_dir / "case.json").read_text())
config = json.loads((input_dir / "mechanism.json").read_text())
design_settings = json.loads((input_dir / "design.json").read_text())

estimation_settings = json.loads((input_dir / "estimation.json").read_text())
# EXAMPLES ONLY: uncomment and choose values appropriate for your model.
# case["operation"]["runtime"]["value"] = 12.0
# case["biology"]["conditions"]["ph"] = 7.0  # culture provider only
# config["state"]["species_mol"]["glucose"] = 0.020  # pathway model only
# design_settings["criterion"] = "A"  # D, A, or E
# Event times use numerics.step.unit. Fixed-step observations must lie on its mesh.
# A new volume also requires matching initial_mass_kg, mechanism state volume,
# and physically consistent species/cell inventories.
# For new settings, review acceptance limits rather than weakening failed checks.
print(json.dumps(case, indent=2))
print("Design parameters:", design_settings["parameters"])
print("Candidate experiments:", design_settings["candidates"])


{
  "biology": {
    "conditions": {},
    "mechanism": "pathway-lp"
  },
  "numerics": {
    "jacobian_relative_step": 1e-06,
    "jacobian_state_scale": 1e-06,
    "absolute_tolerance": 1e-11,
    "backend": "scipy",
    "maximum_step": {
      "unit": "s",
      "value": 30.0
    },
    "relative_tolerance": 1e-08,
    "step": {
      "unit": "h",
      "value": 0.05
    }
  },
  "operation": {
    "carrier_density_kg_l": 1.0,
    "carrier_species": "water",
    "initial_mass_kg": 1.0,
    "initial_volume": {
      "unit": "L",
      "value": 1.0
    },
    "mode": "batch",
    "runtime": {
      "unit": "h",
      "value": 10.0
    }
  },
  "recipe": {
    "name": "batch"
  }
}
Design parameters: [{'name': 'glucose_uptake_max', 'mechanism_path': 'model.parameters.glucose_uptake_max', 'initial_value': 9.5, 'lower_bound': 8.0, 'upper_bound': 12.0, 'relative_step': 0.0001}]
Candidate experiments: [{'identifier': 'low-loading', 'values': {'substrate_mol': 0.0002}, 'cost': 0.0}, {'ident

## Step 3 — Construct and run the native reactor

Run this cell to build the LiquidPhase, biological mechanism, and native reactor and execute the declared schedule. histories retains every segment; unit.result alone is only the latest segment. Use a fresh assembly after configuration changes.

In [ ]:
# STEP 3: Construct and run the native reactor
# Run this cell to build the LiquidPhase, biological mechanism, and native
# reactor and execute the declared schedule. histories retains every segment;
# unit.result alone is only the latest segment. Use a fresh assembly after configuration
# changes.

model = config["model"]
names = tuple(model["state_ids"])
assembly = build_bioreactor(case, config, thermo_path)
unit = assembly.unit
histories = assembly.solve(verbose=False)
molecular_weights = dict(zip(assembly.phase.name_species, assembly.phase.mw))


## Step 4 — Extract trajectories with explicit units

Run to convert recorded inventories using the configured constant density at each time. The display table keeps pre-event boundary rows; histories retains pre/post states for observation selection. Change output quantities here when adapting the model. Do not use these conversions for variable-density physics without supplying a suitable volume history.

In [ ]:
# STEP 4: Extract trajectories with explicit units
# Run to convert recorded inventories using the configured constant density
# at each time. The display table keeps pre-event boundary rows; histories retains
# pre/post states for observation selection. Change output quantities here when adapting
# the model. Do not use these conversions for variable-density physics without supplying
# a suitable volume history.

names = tuple(config["model"]["state_ids"])
rows = []
raw_minimum = float("inf")
for segment_index, result in enumerate(histories):
    mass_history = np.asarray(result.mass_j_liquid0)
    biomass_history = np.asarray(result.biomass_kg_liquid0)
    raw_minimum = min(raw_minimum, float(np.min(mass_history)))
    # Keep the pre-event boundary in this presentation table; histories keeps both sides.
    for i in range(0 if segment_index == 0 else 1, len(result.time)):
        mass = mass_history[i]
        volume = mass.sum() / (case["operation"]["carrier_density_kg_l"] * 1000.0)
        row = {"time_h": result.time[i] / 3600.0}
        for name in names:
            j = list(assembly.phase.name_species).index(name)
            row[f"{name}_mmol_l"] = mass[j] / molecular_weights[name] * 1e6 / (volume * 1000.0)
        row["biomass_gdw_l"] = biomass_history[i] / volume
        rows.append(row)

print("Recorded rows:", len(rows))
print("Final row:", rows[-1])


## Step 5 — Check, summarize, and validate forward results

Inspect finiteness, nonnegative inventories, and the optional original endpoint regression. These checks are not complete elemental conservation or experimental validation. NOT_CHECKED means no original-example comparison was made.

***Note***: The canonical comparison is only applicable to the unchanged distributed example. Finiteness and inventory checks remain applicable to modified or new simulations.

In [ ]:
# STEP 5: Check, summarize, and validate forward results
# Inspect finiteness, nonnegative inventories, and the optional original
# endpoint regression. These checks are not complete elemental conservation or
# experimental validation. NOT_CHECKED means no original-example comparison was made.

# Compare with distributed evidence only when every original input is unchanged.
configured_documents = {"case.json": case, "mechanism.json": config, "design.json": design_settings, "estimation.json": estimation_settings}
REFERENCE_INPUT_SHA256 = {
    "case.json": "71bc288d8d5a1cb0e71fc9e6bba08248c203a9341a5dc0a75f83b61109e77641",
    "mechanism.json": "c50f4f8c8c2aa5580d41de7daa45ba9d5be6bdefcb17a0bfa0f4fc90ee072015",
    "design.json": "cefeb04a19f3e4ada8cbee590e2036d4bf1484c2513df37d30b0621252d38da0",
    "estimation.json": "ca03c824d010d1b59e5eadb30fbc37288a3d17ff02622e189751bf41c72eed7d",
    "thermo.json": "c5e7916f6ebb5e70ebd5800d3b73070d11881e669adf129d6e81e151bf12f0eb"
}
reference_inputs_match = all(
    hashlib.sha256(json.dumps(document, sort_keys=True).encode()).hexdigest()
    == REFERENCE_INPUT_SHA256[name]
    for name, document in configured_documents.items()
) and hashlib.sha256(thermo_path.read_bytes()).hexdigest() == REFERENCE_INPUT_SHA256["thermo.json"]
check_regression = COMPARE_CANONICAL and reference_inputs_match
print("Original-example regression:", "ENABLED" if check_regression else "NOT_CHECKED")

baseline = (json.loads((reference_dir / 'inputs/regression_baseline.json').read_text())
            if check_regression else None)
expected_biomass = (baseline['endpoints']['final_biomass_gdw_l'] if baseline
                    else rows[-1]['biomass_gdw_l'])
biomass_error = (abs(rows[-1]["biomass_gdw_l"] - expected_biomass) / expected_biomass
                 if baseline else None)

status = ('NOT_CHECKED' if baseline is None else
          'PASS' if biomass_error < baseline['relative_tolerance'] else 'FAIL')
summary = {"status": status, "native_unit": type(unit).__name__,
           "final_biomass_gdw_l": rows[-1]["biomass_gdw_l"],
           "reference_final_biomass_gdw_l": expected_biomass,
           "relative_final_biomass_error": biomass_error,
           "raw_minimum_species_mass_kg": raw_minimum,
           "comparison_kind": "generated_model_regression_not_experimental_validation"}

balance_audit = {"finite": bool(np.isfinite(np.array([list(row.values()) for row in rows])).all()), "raw_minimum_species_mass_kg": raw_minimum, "nonnegative_inventory_constraint": raw_minimum >= -1e-12}
assert balance_audit["finite"]
assert balance_audit.get("nonnegative_inventory_constraint", balance_audit.get("passed"))
if check_regression:
    assert summary["status"] == "PASS", "forward endpoint regression failed"
simulation = {"summary": summary, "rows": rows}
print(json.dumps(summary, indent=2))
print(balance_audit)


: 

## Step 6 — Plot forward outputs

Run to display the native simulation results. Adjust species selection and labels for your case; underlying balances are unchanged. The two examples use different output bases; inspect axis labels before comparison.

In [ ]:
# STEP 6: Plot forward outputs
# Run to display the native simulation results. Adjust species selection and
# labels for your case; underlying balances are unchanged. The two examples use
# different output bases; inspect axis labels before comparison.

figure, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True, constrained_layout=True)
time = [row["time_h"] for row in rows]
for name in names:
    axes[0].plot(time, [row[f"{name}_mmol_l"] for row in rows], label=name)
axes[1].plot(time, [row["biomass_gdw_l"] for row in rows], label="biomass")
axes[0].set(ylabel="Species (mmol/L)", title="Native pathway-model trajectory")
axes[1].set(xlabel="Time (h)", ylabel="Biomass (gDW/L)")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.show()


## Step 7 — Define the estimation observation adapter

This visible callback rebuilds the pathway batch model for every trial parameter vector, maps hours to solver seconds, and extracts glucose/biomass observations. It supports this batch pathway template, not arbitrary scheduled cultures; use native_experiment_model with explicit event-side observation declarations for those. To use measured data, replace the synthetic generation in the next cell and remove truth-based assessment.

In [ ]:
# STEP 7: Define the estimation observation adapter
# This visible callback rebuilds the pathway batch model for every trial
# parameter vector, maps hours to solver seconds, and extracts glucose/biomass
# observations. It supports this batch pathway template, not arbitrary scheduled
# cultures; use native_experiment_model with explicit event-side observation
# declarations for those. To use measured data, replace the synthetic generation in the
# next cell and remove truth-based assessment.

def trajectory_model(case, definition, thermo_path, settings, *, tighter=False):
    """Return a callback that integrates a fresh reactor at each trial parameter set."""
    scales = {"s": 1.0, "h": 3600.0, "day": 86400.0}
    seconds = scales[settings["time_unit"]]

    def predict(parameters, times):
        times = np.asarray(times, dtype=float)
        if times.ndim != 1 or not len(times) or not np.isfinite(times).all() or np.any(times < 0):
            raise ValueError("observation times must be finite and nonnegative")
        configured = deepcopy(definition)
        for name, value in parameters.items():
            if name not in configured["model"]["parameters"]:
                raise ValueError(f"unknown model parameter {name!r}")
            configured["model"]["parameters"][name] = value
        operation = deepcopy(case)
        operation["numerics"].update(settings["integration_tolerances"])
        if tighter:
            operation["numerics"]["relative_tolerance"] *= 0.1
            operation["numerics"]["absolute_tolerance"] *= 0.1
        assembly = build_bioreactor(operation, configured, thermo_path)
        requested = times * seconds
        if requested.max() > assembly.time_grid[-1]:
            raise ValueError("observation time exceeds declared reactor runtime")
        assembly.time_grid = np.unique(np.r_[0.0, requested])
        if assembly.time_grid[-1] == 0:
            raise ValueError("trajectory fitting requires a positive observation time")
        assembly.solve(verbose=False)
        result = assembly.unit.result
        positions = np.searchsorted(result.time, requested)
        if not np.allclose(np.asarray(result.time)[positions], requested, rtol=0, atol=1e-8):
            raise RuntimeError("solver did not return requested observation times")
        mass = np.asarray(result.mass_j_liquid0)[positions]
        volume_m3 = mass.sum(axis=1) / (case["operation"]["carrier_density_kg_l"] * 1000.0)
        outputs = {}
        for item in settings["measurements"]:
            if item["quantity"] == "biomass_concentration":
                if item["unit"] != "gDW/L":
                    raise ValueError("biomass observations require gDW/L")
                values = np.asarray(result.biomass_kg_liquid0)[positions] / volume_m3
            elif item["quantity"] == "species_concentration":
                if item["unit"] != "mmol/L":
                    raise ValueError("species observations require mmol/L")
                j = list(assembly.phase.name_species).index(item["species"])
                values = mass[:, j] / assembly.phase.mw[j] * 1000.0 / volume_m3
            else:
                raise ValueError("unsupported observation quantity")
            outputs[item["name"]] = values
        return outputs

    return predict


## Step 8 — Prepare estimation observations and bounds

Edit estimation_settings in Step 2 to select parameters, covariance, training times, and withheld times. This cell generates seeded synthetic data and passes only training observations to the estimator. For real data, supply observed arrays in matching units; synthetic truth is not an optimizer input.

In [ ]:
# STEP 8: Prepare estimation observations and bounds
# Edit estimation_settings in Step 2 to select parameters, covariance,
# training times, and withheld times. This cell generates seeded synthetic data and
# passes only training observations to the estimator. For real data, supply observed
# arrays in matching units; synthetic truth is not an optimizer input.

settings = estimation_settings
definition = config
declarations = settings["parameters"]
train = np.asarray(settings["training_times"], dtype=float)
held = np.asarray(settings["held_out_times"], dtype=float)
if not len(train) or not len(held) or np.intersect1d(train, held).size:
    raise ValueError("training and held-out times must be nonempty and disjoint")
times = np.r_[train, held]
model = trajectory_model(case, definition, thermo_path, settings)
truth = settings["synthetic_data"]["true_parameters"]
exact = model(truth, times)
measurements = settings["measurements"]
covariance = np.asarray(settings["covariance"], dtype=float)
rng = np.random.default_rng(settings["synthetic_data"]["seed"])
noise = rng.multivariate_normal(np.zeros(len(measurements)), covariance, len(times))
observed = {item["name"]: exact[item["name"]] + noise[:, j]
            for j, item in enumerate(measurements)}
observations = tuple(ObservationSeries(
    item["name"], item["unit"], train, observed[item["name"]][:len(train)],
    provenance="seeded synthetic native reactor trajectory; training subset",
) for item in measurements)
problem = BioreactorEstimationProblem(
    parameter_names=[item["name"] for item in declarations],
    initial_values=[item["initial_value"] for item in declarations],
    lower_bounds=[item["lower_bound"] for item in declarations],
    upper_bounds=[item["upper_bound"] for item in declarations],
    observations=observations, covariance=covariance,
)


## Step 9 — Fit the selected parameters

Run bounded least squares through PharmaPy estimation residuals. Each trial simulates a fresh reactor. Inspect convergence and fitted bounds before interpreting uncertainty.

In [ ]:
# STEP 9: Fit the selected parameters
# Run bounded least squares through PharmaPy estimation residuals. Each
# trial simulates a fresh reactor. Inspect convergence and fitted bounds before
# interpreting uncertainty.

result = estimate_bioreactor_parameters(
    problem, model, finite_difference_step=settings["finite_difference_step"],
)

print(dict(result.estimates))


## Step 10 — Assess withheld predictions and solver stability

Run the tighter integration and the unchanged example acceptance checks. Recovery against known truth is meaningful only for synthetic data. For real measurements retain withheld-prediction checks and replace truth-based checks with justified criteria.

In [ ]:
# STEP 10: Assess withheld predictions and solver stability
# Run the tighter integration and the unchanged example acceptance checks.
# Recovery against known truth is meaningful only for synthetic data. For real
# measurements retain withheld-prediction checks and replace truth-based checks with
# justified criteria.

predicted = model(result.estimates, times)
tighter = trajectory_model(case, definition, thermo_path, settings, tighter=True)
refined = tighter(result.estimates, times)
metrics = {}
for j, item in enumerate(measurements):
    name = item["name"]
    residual = (predicted[name] - observed[name]) / np.sqrt(covariance[j, j])
    metrics[name] = {
        "training_standardized_rmse": float(np.sqrt(np.mean(residual[:len(train)] ** 2))),
        "held_out_standardized_rmse": float(np.sqrt(np.mean(residual[len(train):] ** 2))),
        "solver_difference_in_noise_sd": float(np.max(abs(refined[name] - predicted[name])) / np.sqrt(covariance[j, j])),
    }
errors = {name: abs(result.estimates[name] - value) / abs(value) for name, value in truth.items()}
limits = settings["acceptance"]
checks = {
    "optimizer_success": result.success,
    "local_identifiability": result.identifiable,
    "parameter_recovery": all(value <= limits["parameter_relative_error"] for value in errors.values()),
    "training_prediction": all(m["training_standardized_rmse"] <= limits["standardized_rmse"] for m in metrics.values()),
    "held_out_prediction": all(m["held_out_standardized_rmse"] <= limits["standardized_rmse"] for m in metrics.values()),
    "solver_stability": all(m["solver_difference_in_noise_sd"] <= limits["solver_difference_in_noise_sd"] for m in metrics.values()),
}
report = {
    "status": "PASS" if all(checks.values()) else "FAIL", "checks": checks,
    "estimate": result.estimates[problem.parameter_names[0]],
    "estimates": dict(result.estimates), "parameter_relative_errors": errors,
    "covariance": result.covariance.tolist(), "identifiable": result.identifiable,
    "jacobian_rank": result.jacobian_rank, "objective": result.objective,
    "evaluations": result.evaluations, "native_estimator": "ParameterEstimation",
    "native_unit": "BatchReactor", "time_unit": settings["time_unit"],
    "synthetic_data": settings["synthetic_data"], "acceptance": limits,
    "integration_tolerances": settings["integration_tolerances"],
    "metrics": metrics,
    "observations": [{**item, "sample_times": times.tolist(),
        "split": ["training"] * len(train) + ["held_out"] * len(held),
        "values": observed[item["name"]].tolist(),
        "noise_free_values": exact[item["name"]].tolist(),
        "predictions": predicted[item["name"]].tolist()} for item in measurements],
    "evidence_limit": "Synthetic same-model recovery and withheld-time prediction; not independent experimental validation. Covariance is a local residual-scaled approximation.",
}

estimation = report
assert estimation["status"] == "PASS", "dynamic estimation acceptance failed"
print(json.dumps({"status": report["status"], "estimates": report["estimates"], "checks": report["checks"]}, indent=2))


## Step 11 — Build fresh simulation callbacks for experiment design

Review design_settings from Step 2: parameter paths, candidate control paths, cost limits, observations/covariance, pre/post event timing, training/withheld times, seeds, and acceptance profile. Fixed-step sample times must lie on the mesh. This shared library adapter constructs a new native experiment per evaluation; no example runner is used.

In [ ]:
# STEP 11: Build fresh simulation callbacks for experiment design
# Review design_settings from Step 2: parameter paths, candidate control
# paths, cost limits, observations/covariance, pre/post event timing, training/withheld
# times, seeds, and acceptance profile. Fixed-step sample times must lie on the mesh.
# This shared library adapter constructs a new native experiment per evaluation; no
# example runner is used.

design_model = native_experiment_model(case, config, thermo_path, design_settings)
refined_design_model = native_experiment_model(
    case, config, thermo_path, design_settings,
    refinement=design_settings["refinement_factor"],
)
print("Criterion:", design_settings["criterion"])
print("Measurements:", design_settings["observations"])
print("Sample times (s):", design_settings["sample_times_s"])
print("Acceptance:", design_settings["acceptance"])


## Step 12 — Calibrate, select an experiment, and refit

Execute the full study; allow tens of minutes. The library coordinator fits baseline synthetic observations, computes finite-difference information, ranks candidates, replays selection, generates paired selected/control observations, refits, and checks numerical refinement. It does not request laboratory measurements. The true synthetic parameter values generate assessment data; selection uses fitted values.

In [ ]:
# STEP 12: Calibrate, select an experiment, and refit
# Execute the full study; allow tens of minutes. The library coordinator
# fits baseline synthetic observations, computes finite-difference information, ranks
# candidates, replays selection, generates paired selected/control observations, refits,
# and checks numerical refinement. It does not request laboratory measurements. The true
# synthetic parameter values generate assessment data; selection uses fitted values.

design = run_design_study(design_settings, design_model, refined_design_model)
assert design["status"] == "PASS", "experiment-design acceptance failed"
print(json.dumps({key: design[key] for key in (
    "status", "selected_id", "checks", "capability_status", "empirical_benefit_status",
    "predicted_sd_reduction_vs_matched_control", "mean_selected_prediction_mse",
    "mean_control_prediction_mse",
)}, indent=2))


**Interpretation:** capability acceptance and realized predictive benefit are separate. The CHO reference study reports `NOT_DEMONSTRATED` empirical benefit despite improved predicted precision. Do not substitute a favorable criterion, seed, or tolerance after seeing results. To fit real experiments, use `ObservationSeries` and `BioreactorEstimationProblem`; this combined coordinator specifically generates synthetic demonstration data.

## Step 13 — Compare the unchanged example with distributed evidence

This optional gate retains the original regression tolerances and checks. It only runs for unchanged inputs; custom cases receive NOT_CHECKED for reference agreement while study acceptance remains mandatory. Do not regenerate the original baseline just to make changed biology pass.

In [ ]:
# STEP 13: Compare the unchanged example with distributed evidence
# This optional gate retains the original regression tolerances and checks.
# It only runs for unchanged inputs; custom cases receive NOT_CHECKED for reference
# agreement while study acceptance remains mandatory. Do not regenerate the original
# baseline just to make changed biology pass.

if check_regression:
    canonical = reference_dir / "outputs"
    generated_trajectory = simulation["rows"]
    canonical_trajectory = np.genfromtxt(canonical / "trajectories.csv", delimiter=",", names=True)
    trajectory_agrees = all(np.allclose([row[name] for row in generated_trajectory], canonical_trajectory[name]) for name in canonical_trajectory.dtype.names)
    canonical_summary = json.loads((canonical / "summary.json").read_text())
    canonical_estimation = json.loads((canonical / "parameter_estimation.json").read_text())
    metrics_agree = (np.isclose(simulation["summary"]["relative_final_biomass_error"], canonical_summary["relative_final_biomass_error"]) and np.isclose(estimation["estimate"], canonical_estimation["estimate"]))
    canonical_design = json.loads((canonical / "experiment_design.json").read_text())
    assert design["status"] == design["capability_status"] == "PASS"
    assert all(design["checks"][name] for name in design["required_checks"])
    assert design["acceptance_profile"] == canonical_design["acceptance_profile"]
    assert design["empirical_benefit_status"] == canonical_design["empirical_benefit_status"]
    assert design["selected_id"] == canonical_design["selected_id"]
    np.testing.assert_allclose(design["score_improvement"], canonical_design["score_improvement"], rtol=1e-4, atol=1e-8)
    workflow_result = {"status": "PASS" if trajectory_agrees and metrics_agree else "FAIL", "native_unit": simulation["summary"]["native_unit"], "simulation_status": simulation["summary"]["status"], "estimation_status": estimation["status"], "design_status": design["status"], "trajectory_agrees": bool(trajectory_agrees), "metrics_agree": bool(metrics_agree)}
    assert simulation["summary"]["status"] == estimation["status"] == "PASS"
    assert workflow_result["status"] == "PASS"
    workflow_result.update(acceptance_profile=design["acceptance_profile"], capability_status=design["capability_status"], empirical_benefit_status=design["empirical_benefit_status"])
    print('WORKFLOW_RESULT=' + json.dumps(workflow_result, sort_keys=True))
else:
    workflow_result = {"status": "PASS", "regression_status": "NOT_CHECKED",
                       "trajectory_agrees": None, "metrics_agree": None,
                       "native_unit": simulation["summary"]["native_unit"],
                       "simulation_status": simulation["summary"]["status"],
                       "design_status": design["status"],
                       "capability_status": design["capability_status"],
                       "empirical_benefit_status": design["empirical_benefit_status"]}
    print('WORKFLOW_RESULT=' + json.dumps(workflow_result, sort_keys=True))


## Step 14 — Optionally export this run

Default is no file writes. Set WRITE_ARTIFACTS=True in Step 1 and choose EXPORT_DIR to save a separate report and CSV; distributed outputs are protected. Segment mass files preserve both sides of material events. Copy your edited input dictionaries too so the run can be reproduced.

In [ ]:
# STEP 14: Optionally export this run
# Default is no file writes. Set WRITE_ARTIFACTS=True in Step 1 and choose
# EXPORT_DIR to save a separate report and CSV; distributed outputs are protected.
# Segment mass files preserve both sides of material events. Copy your edited input
# dictionaries too so the run can be reproduced.

if WRITE_ARTIFACTS:
    output = EXPORT_DIR.resolve()
    protected = (reference_dir / "outputs").resolve()
    if output == protected or protected in output.parents:
        raise ValueError("Choose an export folder outside distributed reference outputs")
    output.mkdir(parents=True, exist_ok=True)
    with (output / "trajectories.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=simulation["rows"][0])
        writer.writeheader()
        writer.writerows(simulation["rows"])
    reports = {"summary": simulation["summary"], "balance_audit": balance_audit,
               "experiment_design": design, "workflow_result": workflow_result,
               "validation": {"status": simulation["summary"]["status"],
                              "independent_experimental_validation": False,
                              "criterion": "Original-model regression only when inputs match"}}
    if "estimation" in globals():
        reports["parameter_estimation"] = estimation
    for name, report in reports.items():
        (output / f"{name}.json").write_text(json.dumps(report, indent=2) + "\n")
    saved_inputs = output / "inputs"
    saved_inputs.mkdir(exist_ok=True)
    for name, document in configured_documents.items():
        (saved_inputs / name).write_text(json.dumps(document, indent=2) + "\n")
    (saved_inputs / "thermo.json").write_bytes(thermo_path.read_bytes())
    for index, segment in enumerate(histories):
        data = np.column_stack((segment.time, segment.mass_j_liquid0))
        np.savetxt(output / f"species_segment_{index:03d}.csv", data, delimiter=",",
                   header=",".join(["time_s"] + [f"{name}_kg" for name in assembly.phase.name_species]), comments="")
    figure.savefig(output / "trajectories.png", dpi=180)
    print("Saved run to", output)
else:
    print("Results remain in memory; no artifacts written.")
plt.close(figure)
